# Rhon Data Cleaning & Keyword Filter

Cleans the raw Apify export, outputs a structured file, and then extracts comments that reference Kenyan gender or economic norms using a curated keyword list.

In [1]:
import pandas as pd
import re

In [8]:
import pandas as pd

# Filepaths (update if necessary)
scored_path = "/Users/sushildalavi/Desktop/NLC/WBProj/rhonairobi_scored.xlsx"
yt_path     = "/Users/sushildalavi/Desktop/NLC/WBProj/rhon_YTcomments.xlsx"
apify_path  = "/Users/sushildalavi/Desktop/NLC/WBProj/rhon_apify_scraped_cleaned.xlsx"

# Load the datasets
df_scored = pd.read_excel(scored_path)
df_yt     = pd.read_excel(yt_path)
df_apify  = pd.read_excel(apify_path)

# Rename columns so they all share a 'comment' column
df_scored = df_scored.rename(columns={'full_text': 'comment'})
df_yt     = df_yt.rename(columns={'comment': 'comment'})     # already named comment
df_apify  = df_apify.rename(columns={'text': 'comment'})

# Keep only the comment column
scored_comments = df_scored[['comment']].copy()
yt_comments     = df_yt[['comment']].copy()
apify_comments  = df_apify[['comment']].copy()

# Clean and remove missing/blank comments
def clean_and_drop(df):
    # convert to string and strip whitespace
    df['comment'] = df['comment'].astype(str).str.strip()
    # drop rows where comment is NaN or empty
    return df[df['comment'].astype(bool)].dropna(subset=['comment'])

scored_comments_clean = clean_and_drop(scored_comments)
yt_comments_clean     = clean_and_drop(yt_comments)
apify_comments_clean  = clean_and_drop(apify_comments)

print("Original counts:")
print(f"  Scored: {len(scored_comments)}, YouTube: {len(yt_comments)}, Apify: {len(apify_comments)}")
print("Counts after cleaning:")
print(f"  Scored: {len(scored_comments_clean)}, YouTube: {len(yt_comments_clean)}, Apify: {len(apify_comments_clean)}")

# Concatenate datasets
combined = pd.concat(
    [scored_comments_clean, yt_comments_clean, apify_comments_clean],
    ignore_index=True
)

# Drop duplicate comments
combined_dedup = combined.drop_duplicates(subset=['comment'])

# Basic EDA: number of unique comments and comment length stats
print(f"\nCombined unique comments: {len(combined_dedup)}")
combined_dedup['comment_length'] = combined_dedup['comment'].str.len()
print("Comment length statistics:")
print(combined_dedup['comment_length'].describe())

# Save the cleaned & deduplicated comments if you wish
output_path = "/Users/sushildalavi/Desktop/NLC/WBProj/rhon_merged_cleaned.xlsx"
combined_dedup[['comment']].to_excel(output_path, index=False)
print(f"\nCleaned dataset saved to {output_path}")

Original counts:
  Scored: 4106, YouTube: 15745, Apify: 936
Counts after cleaning:
  Scored: 4106, YouTube: 15739, Apify: 936

Combined unique comments: 18429
Comment length statistics:
count    18429.000000
mean        89.164686
std        127.146923
min          1.000000
25%         27.000000
50%         55.000000
75%        110.000000
max       5512.000000
Name: comment_length, dtype: float64


/var/folders/dv/4yw53xh14jg0_mcrwc977cgm0000gn/T/ipykernel_10798/1030504989.py:50: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  combined_dedup['comment_length'] = combined_dedup['comment'].str.len()



Cleaned dataset saved to /Users/sushildalavi/Desktop/NLC/WBProj/rhon_merged_cleaned.xlsx


In [2]:
pip install openai

  Using cached openai-2.6.1-py3-none-any.whl.metadata (29 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached jiter-0.11.1-cp314-cp314-macosx_11_0_arm64.whl.metadata (5.2 kB)
  Using cached pydantic-2.12.3-py3-none-any.whl.metadata (87 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.41.4-cp314-cp314-macosx_11_0_arm64.whl.metadata (7.3 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
Using cached openai-2.6.1-py3-none-any.whl (1.0 MB)
Using cached distro-1.9.0-py3-none-any.whl (20 kB)
Using cached jiter-0.11.1-cp314-cp314-macosx_11_0_arm64.whl (315 kB)
Using cached pydantic-2.12.3-py3-none-any.whl (462 kB)
Using cached pydantic_core-2.41.4-cp314-cp314-macosx_11_0_arm64.whl (1.9 MB)
Using cached annotated_types-0.7.0-py3-none-any.whl (13 kB)
Using cached tqdm-4.67.1-py3-none-any.whl (78 kB)
Using cached typing_in

In [9]:
import pandas as pd

# ======= Load =======
file_path = "/Users/sushildalavi/Desktop/NLC/WBProj/rhon_merged_cleaned.xlsx"
df = pd.read_excel(file_path, dtype=str)

# Drop columns that are completely empty (often "Unnamed: 0" from prior exports)
df = df.dropna(axis=1, how="all")

# If there's already a 'comment' column, use it.
if "comment" in df.columns:
    comment_series = df["comment"].astype(str)
else:
    # If only 1 column remains, treat it as 'comment'
    if len(df.columns) == 1:
        only_col = df.columns[0]
        comment_series = df[only_col].astype(str)
    else:
        # Pick the column with the most non-null values as the comment column (fallback)
        non_null_counts = df.notna().sum().sort_values(ascending=False)
        best_col = non_null_counts.index[0]
        print(f"[INFO] Multiple columns detected {list(df.columns)}; using '{best_col}' as comment.")
        comment_series = df[best_col].astype(str)

# ======= Build a single-column DF named 'comment' =======
df = pd.DataFrame({"comment": comment_series})

# ======= Clean =======
df["comment"] = df["comment"].str.strip()
df = df.dropna(subset=["comment"])
df = df[df["comment"] != ""]

before = len(df)
df = df.drop_duplicates(subset=["comment"])
print(f"Removed {before - len(df)} duplicate comments.")

# ======= EDA =======
print(f"\n✅ Final rows: {len(df)}")

# Comment length stats
df["comment_length"] = df["comment"].apply(len)
print("\nComment length statistics:")
print(df["comment_length"].describe())

# Missing values
print("\nMissing values per column:")
print(df.isnull().sum())

# Sample rows
print("\nSample comments:")
print(df["comment"].head(10).to_string(index=False))

# ======= Save cleaned, rerank-ready =======
output_path = "/Users/sushildalavi/Desktop/NLC/WBProj/rhon_ready_for_rerank.xlsx"
df[["comment"]].to_excel(output_path, index=False)
print(f"\n💾 Cleaned dataset saved to: {output_path}")

Removed 1 duplicate comments.

✅ Final rows: 18428

Comment length statistics:
count    18428.000000
mean        89.140167
std        127.106796
min          1.000000
25%         27.000000
50%         55.000000
75%        109.250000
max       5512.000000
Name: comment_length, dtype: float64

Missing values per column:
comment           0
comment_length    0
dtype: int64

Sample comments:
Zozibini Tunzi 🔥🔥\n\n#R83Million Satanism #RHON...
Zip Up Bag\n\nKsh.2200 ✅ +254 701 131076\n\n#ba...
     Zennah and Farrah need to make up #RHONairobi
Zenna bringing it on #RHONairobi https://t.co/l...
Zenah’s makeup 😭😭 in her yellow confessional lo...
Zenah was baptised by fire in the wrong way. Yo...
Zenah shouldn't have come to this trip mos. #RH...
Zenah is right about mine only apologizing beca...
Zenah is busy calling out mistakes but can’t ac...
Zenah has her issues but since the beginning of...

💾 Cleaned dataset saved to: /Users/sushildalavi/Desktop/NLC/WBProj/rhon_ready_for_rerank.xlsx


In [3]:
pip install python-dotenv


  Using cached python_dotenv-1.2.1-py3-none-any.whl.metadata (25 kB)
Using cached python_dotenv-1.2.1-py3-none-any.whl (21 kB)
Note: you may need to restart the kernel to use updated packages.


In [12]:
from dotenv import load_dotenv
import os
from openai import OpenAI

load_dotenv()  # reads the .env file and sets environment variables

api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)


In [14]:
import os
import re
import json
import time
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

# ================== Config ==================
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

# File paths for the Real Housewives of Nairobi project
INPUT_FILE   = "/Users/sushildalavi/Desktop/NLC/WBProj/rhon_merged_cleaned.xlsx"
FILTERED_FILE = "/Users/sushildalavi/Desktop/NLC/WBProj/rhon_gender_filtered_rerank.xlsx"
FINAL_FILE    = "/Users/sushildalavi/Desktop/NLC/WBProj/rhon_gender_classified.xlsx"

# Toggle this to True if you want to run the LLM classification step
USE_OPENAI_CLASSIFIER = False  # ← set True to use the API

if USE_OPENAI_CLASSIFIER and not api_key:
    raise ValueError("OPENAI_API_KEY not found. Set it in your .env or disable USE_OPENAI_CLASSIFIER.")

client = OpenAI(api_key=api_key) if (USE_OPENAI_CLASSIFIER and api_key) else None

# ================== Load & Normalize ==================
df = pd.read_excel(INPUT_FILE, dtype=str)

# Drop entirely empty columns (e.g., "Unnamed: 0")
df = df.dropna(axis=1, how="all")

# Build a single 'comment' column, regardless of original shape/headers
if "comment" in df.columns:
    comment_series = df["comment"].astype(str)
elif "text" in df.columns:
    comment_series = df["text"].astype(str)
elif len(df.columns) == 1:
    # If the sheet truly has just one column (possibly unnamed), treat it as comment
    only_col = df.columns[0]
    comment_series = df[only_col].astype(str)
else:
    # Fallback: pick the most populated column as comment
    best_col = df.notna().sum().sort_values(ascending=False).index[0]
    print(f"[INFO] Multiple columns detected {list(df.columns)}; using '{best_col}' as the comment column.")
    comment_series = df[best_col].astype(str)

df = pd.DataFrame({"comment": comment_series})

# Basic clean: strip, drop blanks/NaNs, drop exact duplicates
df["comment"] = df["comment"].str.strip()
df = df.dropna(subset=["comment"])
df = df[df["comment"] != ""]
df = df.drop_duplicates(subset=["comment"]).reset_index(drop=True)

print(f"Loaded {len(df)} comments after normalization/cleaning.")

# ================== Keywords (Kenya / Gender) ==================
gender_keywords = [
    # 1. Kenya Local Culture — English
    "kenya", "tribe", "tribal", "norms", "cultural norms", "social norms", "religious norms", "taboo",
    "stigma", "ritual", "ceremony", "rites of passage", "tradition", "traditional beliefs", "traditional values",
    "custom", "sin", "marriage", "bride", "bride price", "child marriage", "arranged marriage",
    "superstition", "curse", "witchcraft", "village", "rural", "community", "elder", "ancestors",
    "traditional chief",

    # 1. Kenya Local Culture — Religious (English)
    "christianity", "church", "islam", "muslim", "imam", "pastor",

    # 1. Kenya Local Culture — Language/Ethnic markers (English)
    "swahili", "sheng", "kikuyu", "luo", "luhya", "kamba",

    # 2.1 Gender Norms & Dynamics — General Norms (English)
    "gender equality", "gender roles", "gender norms", "gender stereotypes", "gender bias",
    "gender sensitivity", "patriarchy", "toxic masculinity", "chauvinism", "feminism", "feminist",
    "women", "girls", "men", "boys", "husband", "wife", "single mother", "breadwinner", "provider",
    "as a woman", "as a man", "women should", "women shouldn’t", "men should", "men shouldn’t",
    "girls should", "girls shouldn’t", "boys should", "boys shouldn’t", "supposed to", "expected to",
    "ought to", "permission", "gatekeeper", "domestic duties", "house chores", "dependent", "emotional",
    "weak", "strong", "man of the house", "purity", "virgin", "promiscuity", "sugar daddy",
    "victim-blaming", "gender-based violence", "sexual harassment", "gender oppression", "equity",
    "advocacy", "inclusivity", "femicide",

    # 2.1 Gender Norms & Dynamics — Vulgarity / Objectification
    "bitch", "slut", "whore",
    "malaya", "kuma", "mbwa", "mjinga", "nyash",

    # 2.1 Gender Norms & Dynamics — Femininity Stereotypes
    "good girl", "beauty queen", "mke mzuri", "msichana mzuri",
    "mama mboga", "shosho", "cucu", "slay queen",

    # 2.1 Gender Norms & Dynamics — Masculinity Stereotypes
    "real man", "alpha male", "simp", "mwanaume", "bazu",
    "kisii man", "mkisii", "kukaliwa", "kufinywa",

    # 2.1 Gender Norms & Dynamics — Sexuality / Nonconformity
    "sugar mummy", "malaya wa campus", "shoga", "lesbian", "msenge",
    "mpango wa kando", "ben 10", "nyumba ndogo", "kutoa mimba",

    # 2.1 Gender Norms & Dynamics — Body Shaming / Objectification
    "figure 8", "flat", "nyash",

    # 2.1 Gender Norms & Dynamics — Backlash / Neosexism
    "feminism too much", "men silenced",

    # 2.1 Gender Norms & Dynamics — Behavioral Expectations
    "women kitchen", "submit", "man up", "proverbs 31 woman", "mshamba", "mshenzi",

    # 2.1 Gender Norms & Dynamics — Tribal / Intersectional (Kenya-specific)
    "cut", "fgm", "kikuyu woman", "mshenzi", "kamba woman",

    # 2.2 Women’s Economic Empowerment
    "women empowerment", "female empowerment", "economic empowerment",
    "financial independence", "women’s rights", "women’s liberation", "feminist movement",
    "education", "college", "university", "degree", "graduation", "scholarship",
    "career", "job", "employment", "promotion", "networking", "self-made", "entrepreneur",
    "businesswoman", "startup", "income", "salary", "money", "investment", "financial literacy",
    "saving", "microfinance", "microloan", "rank", "leadership", "professional growth",
    "upward mobility", "economic independence", "women leaders", "career advancement",
    "side hustle", "girl boss", "madam boss", "women can have it all", "women in business",
    "merry go round", "remote work", "online businesses", "mama mboga", "mama fua",

    # 2.3 Hashtags & Slang — Activism / Empowerment / Safety
    "#EndGBV", "#EndFGM", "#EndFemicideKE", "#StopKillingWomen", "#WeAreNotSafe",
    "#StopKillingUs", "#ForEveryChild", "#NiSisi", "#SexualHealthDay2025",
    "#SexualHealthMatters", "#SexualHealth4All", "#daysofactivism", "#MyDressMyChoice",
    "#JusticeForKenyansGirls",

    # 2.3 Hashtags & Slang — Gender Based Violence (Swahili)
    "kupigwa", "kubakwa"
]

# Build a regex that’s case-insensitive and handles word-ish boundaries.
# We keep hashtags/phrases intact; for phrases we escape & join by '|'.
escaped = [re.escape(k) for k in gender_keywords]
pattern = re.compile(r"(?i)\b(" + "|".join(escaped) + r")\b")

def find_matches(text: str):
    return list({m.group(0) for m in pattern.finditer(text or "")})

# Compute matches and keep only rows with ≥1 match
df["matched_keywords"] = df["comment"].apply(find_matches)
df["gender_flag"] = df["matched_keywords"].apply(lambda lst: len(lst) > 0)

filtered = df[df["gender_flag"]].copy().reset_index(drop=True)
print(f"Filtered down to {len(filtered)} comments matching gender/Kenya keywords.")

# Save filtered for re-ranking input
filtered.to_excel(FILTERED_FILE, index=False)
print(f"Saved filtered set with matches → {FILTERED_FILE}")

# ================== (Optional) LLM Classification ==================
# If you want a lightweight label (e.g., 'Gender_Norms' vs 'Other') from the model,
# set USE_OPENAI_CLASSIFIER=True at the top.
def classify_batch_llm(texts, max_retries=3, sleep_s=1.5):
    """
    Classify each text into one of:
      - 'Gender_Norms' (discusses gender roles/norms/dynamics/empowerment/GBV)
      - 'Local_Culture' (Kenyan cultural/religious customs with no explicit gender framing)
      - 'Other' (none of the above)
    Returns a list of dicts: {"label": str, "rationale": str}
    """
    out = []
    for t in texts:
        attempt = 0
        while True:
            try:
                resp = client.responses.create(
                    model="gpt-4.1-mini",
                    input=[
                        {
                            "role": "system",
                            "content": (
                                "You are a precise annotator. "
                                "Label the input text into one of: "
                                "'Gender_Norms', 'Local_Culture', or 'Other'. "
                                "Return strict JSON: {\"label\": <one_of>, \"rationale\": <short reason>}."
                            ),
                        },
                        {
                            "role": "user",
                            "content": t[:4000]  # safety limit
                        }
                    ],
                    temperature=0.0
                )
                # Extract text from response
                text_out = resp.output_text
                # Attempt to parse as JSON
                try:
                    parsed = json.loads(text_out)
                except Exception:
                    # If model wrapped JSON in prose, try to extract last JSON object
                    m = re.search(r"\{.*\}", text_out, flags=re.S)
                    parsed = json.loads(m.group(0)) if m else {"label": "Other", "rationale": "Parse fallback"}
                # Normalize label
                label = str(parsed.get("label", "Other"))
                if label not in {"Gender_Norms", "Local_Culture", "Other"}:
                    label = "Other"
                rationale = str(parsed.get("rationale", ""))[:500]
                out.append({"label": label, "rationale": rationale})
                break
            except Exception as e:
                attempt += 1
                if attempt >= max_retries:
                    out.append({"label": "Other", "rationale": f"LLM error: {e}"})
                    break
                time.sleep(sleep_s)
    return out

if USE_OPENAI_CLASSIFIER:
    print("Running LLM classification on filtered comments...")
    batch_size = 50
    labels = []
    for i in range(0, len(filtered), batch_size):
        batch = filtered["comment"].iloc[i:i+batch_size].tolist()
        labels.extend(classify_batch_llm(batch))
        print(f"  Classified {min(i+batch_size, len(filtered))}/{len(filtered)}")

    lab_df = pd.DataFrame(labels)
    final = pd.concat([filtered.reset_index(drop=True), lab_df], axis=1)
else:
    # If not classifying, still produce FINAL_FILE mirroring FILTERED_FILE
    final = filtered.copy()
    # Provide a simple rule-based label as a placeholder (optional)
    final["label"] = np.where(final["gender_flag"], "Gender_Norms_or_Local", "Other")
    final["rationale"] = "Rule-based keyword match (no LLM)."

# Save final
final.to_excel(FINAL_FILE, index=False)
print(f"Saved final file → {FINAL_FILE}")

# ================== Tiny EDA printouts ==================
print("\n=== Quick EDA on filtered ===")
final["length"] = final["comment"].str.len()
print(final["length"].describe())

print("\nTop 20 matched keywords by frequency:")
# Flatten matched keywords
from collections import Counter
kw_counter = Counter(k for lst in final["matched_keywords"] for k in lst)
for k, v in kw_counter.most_common(20):
    print(f"{k}: {v}")

Loaded 18428 comments after normalization/cleaning.
Filtered down to 1759 comments matching gender/Kenya keywords.
Saved filtered set with matches → /Users/sushildalavi/Desktop/NLC/WBProj/rhon_gender_filtered_rerank.xlsx
Saved final file → /Users/sushildalavi/Desktop/NLC/WBProj/rhon_gender_classified.xlsx

=== Quick EDA on filtered ===
count    1759.000000
mean      203.717453
std       277.228622
min         6.000000
25%        75.000000
50%       141.000000
75%       250.500000
max      5512.000000
Name: length, dtype: float64

Top 20 matched keywords by frequency:
Kenya: 500
money: 291
women: 181
husband: 116
girls: 87
job: 71
wife: 62
men: 52
kenya: 44
cut: 41
strong: 40
supposed to: 32
University: 29
Money: 22
marriage: 22
KENYA: 19
boys: 17
weak: 16
college: 15
income: 14


In [15]:
import os
import numpy as np
import pandas as pd
from openai import OpenAI
from sklearn.metrics.pairwise import cosine_similarity
import json
import time
from dotenv import load_dotenv

# ============== Setup ==============
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

# UPDATED PATHS (reads your ready-for-rerank one-column sheet)
INPUT_FILE   = "/Users/sushildalavi/Desktop/NLC/WBProj/rhon_ready_for_rerank.xlsx"
FILTERED_FILE = "/Users/sushildalavi/Desktop/NLC/WBProj/rhon_gender_filtered_rerank.xlsx"
FINAL_FILE    = "/Users/sushildalavi/Desktop/NLC/WBProj/rhon_gender_classified.xlsx"

# ============== Load & Normalize ==============
df = pd.read_excel(INPUT_FILE, dtype=str)

# Ensure there is exactly one text column treated as 'comment'
if len(df.columns) != 1:
    raise ValueError(f"Expected exactly 1 column in {INPUT_FILE}, found {len(df.columns)}: {list(df.columns)}")

only_col = df.columns[0]
if only_col != "comment":
    df = df.rename(columns={only_col: "comment"})

# Clean: strip, drop blanks/NaNs, de-dup
df["comment"] = df["comment"].astype(str).str.strip()
df = df.dropna(subset=["comment"])
df = df[df["comment"] != ""].drop_duplicates(subset=["comment"]).reset_index(drop=True)
print(f"Loaded {len(df)} comments.")

# ============== Kenya/RHON keyword list ==============
gender_keywords = [
    # Kenya Local Culture — English
    "kenya", "tribe", "tribal", "norms", "cultural norms", "social norms", "religious norms", "taboo",
    "stigma", "ritual", "ceremony", "rites of passage", "tradition", "traditional beliefs", "traditional values",
    "custom", "sin", "marriage", "bride", "bride price", "child marriage", "arranged marriage",
    "superstition", "curse", "witchcraft", "village", "rural", "community", "elder", "ancestors",
    "traditional chief",
    # Religious (English)
    "christianity", "church", "islam", "muslim", "imam", "pastor",
    # Language/Ethnic markers
    "swahili", "sheng", "kikuyu", "luo", "luhya", "kamba",
    # Gender Norms & Dynamics — General
    "gender equality", "gender roles", "gender norms", "gender stereotypes", "gender bias",
    "gender sensitivity", "patriarchy", "toxic masculinity", "chauvinism", "feminism", "feminist",
    "women", "girls", "men", "boys", "husband", "wife", "single mother", "breadwinner", "provider",
    "as a woman", "as a man", "women should", "women shouldn’t", "men should", "men shouldn’t",
    "girls should", "girls shouldn’t", "boys should", "boys shouldn’t", "supposed to", "expected to",
    "ought to", "permission", "gatekeeper", "domestic duties", "house chores", "dependent", "emotional",
    "weak", "strong", "man of the house", "purity", "virgin", "promiscuity", "sugar daddy",
    "victim-blaming", "gender-based violence", "sexual harassment", "gender oppression", "equity",
    "advocacy", "inclusivity", "femicide",
    # Vulgarity / Objectification
    "bitch", "slut", "whore", "malaya", "kuma", "mbwa", "mjinga", "nyash",
    # Femininity Stereotypes
    "good girl", "beauty queen", "mke mzuri", "msichana mzuri",
    "mama mboga", "shosho", "cucu", "slay queen",
    # Masculinity Stereotypes
    "real man", "alpha male", "simp", "mwanaume", "bazu",
    "kisii man", "mkisii", "kukaliwa", "kufinywa",
    # Sexuality / Nonconformity
    "sugar mummy", "malaya wa campus", "shoga", "lesbian", "msenge",
    "mpango wa kando", "ben 10", "nyumba ndogo", "kutoa mimba",
    # Body Shaming / Objectification
    "figure 8", "flat", "nyash",
    # Backlash / Neosexism
    "feminism too much", "men silenced",
    # Behavioral Expectations
    "women kitchen", "submit", "man up", "proverbs 31 woman", "mshamba", "mshenzi",
    # Tribal / Intersectional
    "cut", "fgm", "kikuyu woman", "mshenzi", "kamba woman",
    # Women’s Economic Empowerment
    "women empowerment", "female empowerment", "economic empowerment", "financial independence",
    "women’s rights", "women’s liberation", "feminist movement", "education", "college",
    "university", "degree", "graduation", "scholarship", "career", "job", "employment",
    "promotion", "networking", "self-made", "entrepreneur", "businesswoman", "startup",
    "income", "salary", "money", "investment", "financial literacy", "saving", "microfinance",
    "microloan", "rank", "leadership", "professional growth", "upward mobility",
    "economic independence", "women leaders", "career advancement", "side hustle",
    "girl boss", "madam boss", "women can have it all", "women in business", "merry go round",
    "remote work", "online businesses", "mama mboga", "mama fua",
    # Activism / Empowerment / Safety Hashtags
    "#EndGBV", "#EndFGM", "#EndFemicideKE", "#StopKillingWomen", "#WeAreNotSafe",
    "#StopKillingUs", "#ForEveryChild", "#NiSisi", "#SexualHealthDay2025",
    "#SexualHealthMatters", "#SexualHealth4All", "#daysofactivism", "#MyDressMyChoice",
    "#JusticeForKenyansGirls",
    # Gender Based Violence (Swahili)
    "kupigwa", "kubakwa"
]

# ============== Embedding-based rerank ==============
print("Running embedding-based rerank...")

comments = df["comment"].tolist()
BATCH = 100
comment_embeddings = []

for i in range(0, len(comments), BATCH):
    batch = comments[i:i+BATCH]
    resp = client.embeddings.create(model="text-embedding-3-large", input=batch)
    batch_embs = [r.embedding for r in resp.data]
    comment_embeddings.extend(batch_embs)
    print(f"Processed {i+len(batch)} / {len(comments)} comments")

comment_embeddings = np.array(comment_embeddings, dtype="float32")

query_text = " ".join(gender_keywords)
query_embedding = client.embeddings.create(
    model="text-embedding-3-large",
    input=[query_text]
).data[0].embedding

print("Computing similarities...")
sims = cosine_similarity(comment_embeddings, [query_embedding]).flatten()

# Keep top-N (cap at dataset size to be safe)
top_n = min(1000, len(df))
top_idx = np.argsort(sims)[-top_n:][::-1]
filtered_df = df.iloc[top_idx].reset_index(drop=True)

print(f"Original: {len(df)} | Retained top {len(filtered_df)}")
filtered_df.to_excel(FILTERED_FILE, index=False)

# ============== Theme classification (LLM) ==============
themes = [
    "Kenya Local Culture",
    "Gender Norms & Dynamics",
    "Gender-Based Violence",
    "Economic Empowerment",
    "Sexual & Reproductive Health",
    "Tribal / Intersectional",
    "Body & Beauty Standards",
    "Emotional States"
]

def classify_comment(comment):
    prompt = f"""
    Classify this comment into one or more of these themes: {themes}.
    Also assign sentiment: Positive, Negative, or Neutral.
    Return JSON only with keys: "themes" (list of strings) and "sentiment" (string).
    Comment: "{comment}"
    """
    try:
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )
        return resp.choices[0].message.content.strip()
    except Exception as e:
        print("Classification error:", e)
        return json.dumps({"themes": [], "sentiment": "Error"})

BATCH_SIZE = 500
classified_rows = []

for start in range(0, len(filtered_df), BATCH_SIZE):
    batch = filtered_df.iloc[start:start+BATCH_SIZE].copy()
    print(f"Processing batch {start//BATCH_SIZE+1}...")
    batch["classification"] = batch["comment"].apply(classify_comment)
    classified_rows.append(batch)
    # Save progress incrementally
    pd.concat(classified_rows, ignore_index=True).to_excel(FINAL_FILE, index=False)
    print(f"Saved progress up to row {start+len(batch)}")

final_df = pd.concat(classified_rows, ignore_index=True)

def parse_classification(text):
    try:
        data = json.loads(text)
        themes_val = ",".join(data.get("themes", []))
        sentiment_val = data.get("sentiment", "")
        return pd.Series([themes_val, sentiment_val])
    except Exception:
        return pd.Series(["", ""])

final_df[["themes", "sentiment"]] = final_df["classification"].apply(parse_classification)
final_df.to_excel(FINAL_FILE, index=False)
print(f"✅ Done! Final classified dataset saved at: {FINAL_FILE}")

Loaded 18428 comments.
Running embedding-based rerank...
Processed 100 / 18428 comments
Processed 200 / 18428 comments
Processed 300 / 18428 comments
Processed 400 / 18428 comments
Processed 500 / 18428 comments
Processed 600 / 18428 comments
Processed 700 / 18428 comments
Processed 800 / 18428 comments
Processed 900 / 18428 comments
Processed 1000 / 18428 comments
Processed 1100 / 18428 comments
Processed 1200 / 18428 comments
Processed 1300 / 18428 comments
Processed 1400 / 18428 comments
Processed 1500 / 18428 comments
Processed 1600 / 18428 comments
Processed 1700 / 18428 comments
Processed 1800 / 18428 comments
Processed 1900 / 18428 comments
Processed 2000 / 18428 comments
Processed 2100 / 18428 comments
Processed 2200 / 18428 comments
Processed 2300 / 18428 comments
Processed 2400 / 18428 comments
Processed 2500 / 18428 comments
Processed 2600 / 18428 comments
Processed 2700 / 18428 comments
Processed 2800 / 18428 comments
Processed 2900 / 18428 comments
Processed 3000 / 18428 c

In [ ]:
import json
import pandas as pd

# ======== Paths (update if needed) ========
INPUT_CLASSIFIED = "/Users/sushildalavi/Desktop/NLC/WBProj/rhon_gender_classified.xlsx"
OUTPUT_RELEVANT  = "/Users/sushildalavi/Desktop/NLC/WBProj/rhon_relevant_comments.xlsx"

# ======== Load ========
df = pd.read_excel(INPUT_CLASSIFIED)

# Normalize column names for safety
df.columns = [c.strip() for c in df.columns]

# ======== Ensure we have 'themes' and 'sentiment' ========
def ensure_parsed_columns(df):
    cols = {c.lower(): c for c in df.columns}
    has_themes = any(c.lower() == "themes" for c in df.columns)
    has_sent   = any(c.lower() == "sentiment" for c in df.columns)
    has_class  = any(c.lower() == "classification" for c in df.columns)

    # If already present, just standardize their names
    if has_themes and has_sent:
        # Standardize to 'themes' and 'sentiment'
        df.rename(
            columns={cols.get("themes", "themes"): "themes",
                     cols.get("sentiment", "sentiment"): "sentiment"},
            inplace=True
        )
        return df

    # Otherwise, try to parse from 'classification' JSON
    if has_class:
        class_col = cols.get("classification", "classification")

        def parse_classification(cell):
            try:
                if isinstance(cell, str):
                    data = json.loads(cell)
                elif isinstance(cell, dict):
                    data = cell
                else:
                    return pd.Series(["", ""])
                themes_val = data.get("themes", [])
                # themes could be list or str; normalize to comma-separated string
                if isinstance(themes_val, list):
                    themes_str = ",".join([str(x).strip() for x in themes_val if str(x).strip()])
                else:
                    themes_str = str(themes_val).strip()
                sentiment_str = str(data.get("sentiment", "")).strip()
                return pd.Series([themes_str, sentiment_str])
            except Exception:
                return pd.Series(["", ""])

        parsed = df[class_col].apply(parse_classification)
        parsed.columns = ["themes", "sentiment"]
        df = pd.concat([df, parsed], axis=1)
        return df

    # If we reach here, there is nothing to parse
    # Create empty columns so filter step will drop everything (as intended)
    df["themes"] = ""
    df["sentiment"] = ""
    return df

df = ensure_parsed_columns(df)

# Ensure we have a 'comment' column; if not, try common fallbacks
if "comment" not in df.columns:
    for cand in ["text", "full_text"]:
        if cand in df.columns:
            df = df.rename(columns={cand: "comment"})
            break

# ======== Filter: keep only relevant, tagged comments ========
# Drop missing/empty themes & sentiment; remove "Error"/"Other"
df = df.dropna(subset=["themes", "sentiment"])
df = df[
    (df["themes"].astype(str).str.strip() != "") &
    (df["sentiment"].astype(str).str.strip() != "") &
    (~df["themes"].str.contains(r"\b(Error|Other)\b", case=False, na=False)) &
    (~df["sentiment"].str.contains(r"\b(Error|Other)\b", case=False, na=False))
]

# Keep only rows that actually have a comment
if "comment" in df.columns:
    df["comment"] = df["comment"].astype(str).str.strip()
    df = df[df["comment"] != ""]
    # De-duplicate by exact comment text
    df = df.drop_duplicates(subset=["comment"])

# ======== (Optional) quick summary ========
if not df.empty:
    theme_counts = (
        df.assign(_theme=df["themes"].astype(str).str.split(","))
          .explode("_theme")
          .assign(_theme=lambda x: x["_theme"].str.strip())
          .query("_theme != ''")
          .groupby("_theme", as_index=False).size()
          .sort_values("size", ascending=False)
    )
    sentiment_counts = df["sentiment"].str.strip().value_counts(dropna=True)

    print("Theme counts (top 20):")
    print(theme_counts.head(20).to_string(index=False))
    print("\nSentiment counts:")
    print(sentiment_counts.to_string())

print(f"\nRemaining relevant comments: {len(df)}")

# ======== Save ========
df.to_excel(OUTPUT_RELEVANT, index=False)
print(f"💾 Saved to: {OUTPUT_RELEVANT}")

Theme counts (top 20):
                      _theme  size
         Kenya Local Culture   225
     Gender Norms & Dynamics   166
     Body & Beauty Standards    75
        Economic Empowerment    68
       Gender-Based Violence    45
            Emotional States    33
     Tribal / Intersectional    17
Sexual & Reproductive Health     7

Sentiment counts:
sentiment
Negative    174
Positive    171
Neutral      71

✅ Remaining relevant comments: 416
💾 Saved to: /Users/sushildalavi/Desktop/NLC/WBProj/rhon_relevant_comments.xlsx


/var/folders/dv/4yw53xh14jg0_mcrwc977cgm0000gn/T/ipykernel_10798/1443364679.py:80: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  (~df["themes"].str.contains(r"\b(Error|Other)\b", case=False, na=False)) &
/var/folders/dv/4yw53xh14jg0_mcrwc977cgm0000gn/T/ipykernel_10798/1443364679.py:81: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  (~df["sentiment"].str.contains(r"\b(Error|Other)\b", case=False, na=False))


In [17]:
import re
from collections import Counter
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from docx import Document
from docx.shared import Inches

# ---------- CONFIG ----------
FILE_PATH = r"/Users/sushildalavi/Desktop/NLC/WBProj/rhon_relevant_comments.xlsx"
OUT_DIR = Path(FILE_PATH).parent / "rhon_figures_pro"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CREATE_DOCX = True
DOCX_PATH = OUT_DIR / "RHON_Visuals.docx"

# ---------- LOAD ----------
df = pd.read_excel(FILE_PATH)
colmap = {c.lower().strip(): c for c in df.columns}

def try_get(*cands):
    for cand in cands:
        k = cand.lower().strip()
        if k in colmap:
            return colmap[k]
    return None

COMMENT_COL  = try_get("comment text", "comment", "text", "comments")
THEMES_COL   = try_get("themes", "theme", "topics")
SENT_TEXTCOL = try_get("sentiment", "sentiments")
# If you have a coded numeric sentiment, include a try_get here (not typical for RHON)

if not COMMENT_COL:
    raise ValueError("Couldn't find the comment column. Please update COMMENT_COL mapping.")

comments = df[COMMENT_COL].dropna().astype(str)

# ---------- MATPLOTLIB STYLE ----------
plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 220,
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})

# ---------- HELPERS ----------
def split_list_cell(val):
    if pd.isna(val):
        return []
    # Split on common delimiters
    parts = re.split(r"[;,/|]+", str(val))
    return [p.strip() for p in parts if p.strip()]

# ---------- SENTIMENT ----------
if SENT_TEXTCOL:
    sentiment = (
        df[SENT_TEXTCOL]
        .astype(str)
        .str.strip()
        .str.capitalize()
        .replace("", pd.NA)
        .dropna()
    )
else:
    # Fallback heuristic from comment text
    pos = re.compile(r"\b(great|amazing|love|powerful|respect|excellent|inspire|brilliant)\b", re.I)
    neg = re.compile(r"\b(terrible|awful|hate|fake|nonsense|trash|worst|cringe|toxic)\b", re.I)
    tmp = []
    for t in comments:
        if pos.search(t): tmp.append("Positive")
        elif neg.search(t): tmp.append("Negative")
        else: tmp.append("Neutral")
    sentiment = pd.Series(tmp)

sent_counts = sentiment.value_counts().to_dict()

# ---------- THEMES ----------
if THEMES_COL:
    themes_series = df[THEMES_COL].apply(split_list_cell)
    all_themes = [th for lst in themes_series for th in lst]
    theme_counts = Counter(all_themes)
else:
    # Fallback theme detection by keywords (tuned for RHON)
    theme_defs = {
        "Gender Norms & Dynamics": r"\b(gender roles|submission|respectability|double standards|patriarchy|masculinity|femininity)\b",
        "Marriage & Relationships": r"\b(marriage|husband|wife|bride|groom|relationship|cheat|loyal|couple)\b",
        "Body & Beauty Standards": r"\b(beauty|body|looks|makeup|skin|weight|fashion|style)\b",
        "Economic Empowerment": r"\b(money|career|business|boss|independent|earn|income|provider)\b",
        "Kenya Local Culture": r"\b(kenya|nairobi|swahili|kikuyu|luo|luhya|culture|tradition)\b",
        "Sexual & Reproductive Health": r"\b(sex|sexual|reproductive|consent|pregnan|abortion|birth|contracep)\b",
        "LGBTQ+ / Sexuality": r"\b(gay|lgbt|queer|same[- ]sex|homophob)\b",
        "Class & Privilege": r"\b(rich|elite|privilege|class|luxury|status)\b",
        "Hypocrisy / Double Standards": r"\b(hypocrisy|double standards|two[- ]faced)\b",
    }
    theme_counts = Counter()
    for t in comments:
        for theme, pat in theme_defs.items():
            if re.search(pat, t, flags=re.I):
                theme_counts[theme] += 1

TOP_N = 10
top_themes = dict(theme_counts.most_common(TOP_N))

# ---------- INFER NORM STANCE (heuristic) ----------
# If you have a dedicated column for stance, map it here instead.
challenge_kw = re.compile(r"\b(patriarchy|double standards|abuse|misogyny|objectify|control|harass|toxic)\b", re.I)
support_kw   = re.compile(r"\b(traditional|should|must|duty|obey|respect elders|culture)\b", re.I)

stance_labels = []
for t in comments:
    if challenge_kw.search(t):
        stance_labels.append("Challenging")
    elif support_kw.search(t):
        stance_labels.append("Supporting")
    else:
        stance_labels.append("Discussing")

stance_counts = Counter(stance_labels)

# ---------- 1) SENTIMENT DONUT ----------
fig = plt.figure(figsize=(6.2, 6.2))
labels = list(sent_counts.keys())
values = list(sent_counts.values())
plt.pie(values, labels=labels, autopct="%1.0f%%", startangle=90)
# donut hole
centre_circle = plt.Circle((0,0), 0.60, fc="white")
fig.gca().add_artist(centre_circle)
plt.title("Sentiment Distribution — RHON")
sent_png = OUT_DIR / "01_sentiment_donut.png"
plt.tight_layout()
plt.savefig(sent_png, bbox_inches="tight")
plt.close()

# ---------- 2) GENDER NORM STANCE (HORIZONTAL BAR + LABELS) ----------
fig = plt.figure(figsize=(7.2, 4.5))
items = sorted(stance_counts.items(), key=lambda x: x[1])  # ascending
labels = [k for k,_ in items]
vals = [v for _,v in items]
ypos = range(len(labels))
plt.barh(list(ypos), vals)
plt.yticks(list(ypos), labels)
for y, v in enumerate(vals):
    plt.text(v + (max(vals) * 0.01 if vals else 0.1), y, str(v), va="center")
plt.xlabel("Count")
plt.title("Attitudes Toward Traditional Gender Norms — RHON")
stance_png = OUT_DIR / "02_gender_norm_stance_barh.png"
plt.tight_layout()
plt.savefig(stance_png, bbox_inches="tight")
plt.close()

# ---------- 3) TOP THEMES LOLLIPOP ----------
fig = plt.figure(figsize=(8.8, 5.2))
items = sorted(top_themes.items(), key=lambda x: x[1])  # ascending
labels = [k for k,_ in items]
vals = [v for _,v in items]
y = range(len(labels))
# stems
for yi, v in zip(y, vals):
    plt.plot([0, v], [yi, yi], linewidth=2)
# markers
plt.scatter(vals, list(y), s=70)
plt.yticks(list(y), labels)
plt.xlabel("Mentions (count)")
plt.title("Top Themes Frequency — RHON")
themes_png = OUT_DIR / "03_themes_lollipop.png"
plt.tight_layout()
plt.savefig(themes_png, bbox_inches="tight")
plt.close()

# ---------- 4) THEME × SENTIMENT HEATMAP ----------
# Build a boolean matrix of theme presence per row (works if THEME_COL exists; else heuristic rebuild)
if THEMES_COL:
    theme_lists = df[THEMES_COL].apply(split_list_cell).tolist()
    theme_names = list(top_themes.keys())
else:
    # Re-derive boolean presence from keyword fallback
    theme_names = list(top_themes.keys())
    # simple substring fallback: presence if theme keyword appears in comment (very lightweight)
    theme_lists = []
    for txt in comments:
        present = []
        for th in theme_names:
            key = th.split("&")[0].split("/")[0].strip().split()[0]
            if re.search(re.escape(key), txt, re.I):
                present.append(th)
        theme_lists.append(present)

sent_series = sentiment.fillna("Unclear").astype(str)
heat = pd.DataFrame(0, index=theme_names, columns=sorted(sent_series.unique()))
for i, ths in enumerate(theme_lists):
    s = sent_series.iloc[i] if i < len(sent_series) else "Unclear"
    for th in ths:
        if th in heat.index and s in heat.columns:
            heat.loc[th, s] += 1

fig = plt.figure(figsize=(9, 6))
ax = plt.gca()
im = ax.imshow(heat.values, aspect="auto")
ax.set_xticks(range(len(heat.columns)))
ax.set_xticklabels(heat.columns, rotation=45, ha="right")
ax.set_yticks(range(len(heat.index)))
ax.set_yticklabels(heat.index)
plt.title("Themes × Sentiment — RHON")
# annotate cells
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        val = int(heat.values[i, j])
        if val:
            ax.text(j, i, str(val), ha="center", va="center")
heat_png = OUT_DIR / "04_theme_by_sentiment_heatmap.png"
plt.tight_layout()
plt.savefig(heat_png, bbox_inches="tight")
plt.close()

# ---------- 5) (OPTIONAL) STACKED BAR: NORM STANCE × SENTIMENT ----------
# Works with the heuristic stance above and the sentiment series
ndf = pd.DataFrame({"stance": stance_labels, "sent": sentiment}).dropna()
pivot = pd.crosstab(ndf["stance"], ndf["sent"]).reindex(
    ["Supporting", "Discussing", "Challenging"]
).fillna(0).astype(int)

fig = plt.figure(figsize=(7.6, 5.0))
bottom = None
for col in pivot.columns:
    plt.bar(pivot.index, pivot[col].values, bottom=bottom, label=col)
    bottom = (pivot[col].values if bottom is None else bottom + pivot[col].values)
plt.ylabel("Count")
plt.title("Norm Stance × Sentiment — RHON")
plt.legend(title="Sentiment", ncol=2)
stack_png = OUT_DIR / "05_normstance_by_sentiment_stacked.png"
plt.tight_layout()
plt.savefig(stack_png, bbox_inches="tight")
plt.close()

print("Saved charts:")
print("-", sent_png)
print("-", stance_png)
print("-", themes_png)
print("-", heat_png)
print("-", stack_png)

# ---------- OPTIONAL: Assemble visuals into a simple Word doc ----------
if CREATE_DOCX:
    doc = Document()
    doc.add_heading("RHON — Visual Insights", level=1)

    doc.add_heading("Sentiment Distribution (Donut)", level=2)
    doc.add_picture(str(sent_png), width=Inches(6.2))

    doc.add_heading("Attitudes Toward Traditional Gender Norms", level=2)
    doc.add_picture(str(stance_png), width=Inches(6.2))

    doc.add_heading("Top Themes Frequency (Lollipop Chart)", level=2)
    doc.add_picture(str(themes_png), width=Inches(6.2))

    doc.add_heading("Themes × Sentiment (Heatmap)", level=2)
    doc.add_picture(str(heat_png), width=Inches(6.2))

    doc.add_heading("Norm Stance × Sentiment (Stacked Bar)", level=2)
    doc.add_picture(str(stack_png), width=Inches(6.2))

    doc.save(DOCX_PATH)
    print(f"Word doc with charts saved to: {DOCX_PATH}")

Saved charts:
- /Users/sushildalavi/Desktop/NLC/WBProj/rhon_figures_pro/01_sentiment_donut.png
- /Users/sushildalavi/Desktop/NLC/WBProj/rhon_figures_pro/02_gender_norm_stance_barh.png
- /Users/sushildalavi/Desktop/NLC/WBProj/rhon_figures_pro/03_themes_lollipop.png
- /Users/sushildalavi/Desktop/NLC/WBProj/rhon_figures_pro/04_theme_by_sentiment_heatmap.png
- /Users/sushildalavi/Desktop/NLC/WBProj/rhon_figures_pro/05_normstance_by_sentiment_stacked.png
Word doc with charts saved to: /Users/sushildalavi/Desktop/NLC/WBProj/rhon_figures_pro/RHON_Visuals.docx


In [23]:
import pandas as pd
from openai import OpenAI
from pathlib import Path
import time

# ---- CONFIG ----
FILE_PATH = r"/Users/sushildalavi/Desktop/NLC/WBProj/RHON YT & X for LLM Coding.xlsx"
MODEL = "gpt-4o"  # Use full GPT-4o for better language understanding
COMMENT_COL_CANDIDATES = ["Comment Text", "comment", "text", "comments"]
SLEEP_BETWEEN_CALLS = 0.4  # seconds to respect rate limits

# ---- SETUP ----
client = OpenAI()

def detect_language(text):
    """Use GPT-4o to detect the language of a comment."""
    prompt = f"""
Detect the primary written language of the following text.
Respond with the language name only (e.g., English, Swahili, French, etc.).
Text:
{text[:500]}
"""
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": "You are an expert linguist identifying languages from social media comments."},
                {"role": "user", "content": prompt}
            ],
            temperature=0
        )
        lang = response.choices[0].message.content.strip()
        return lang if lang else "Unknown"
    except Exception as e:
        print("Error:", e)
        return "Error"

# ---- LOAD DATA ----
print(f"\n🔍 Processing RHON dataset: {FILE_PATH}")
df = pd.read_excel(FILE_PATH)

# Identify comment column
colmap = {c.lower().strip(): c for c in df.columns}
comment_col = None
for cand in COMMENT_COL_CANDIDATES:
    if cand.lower().strip() in colmap:
        comment_col = colmap[cand.lower().strip()]
        break
if not comment_col:
    raise ValueError(f"No recognizable comment column found in {FILE_PATH}")

df[comment_col] = df[comment_col].astype(str).fillna("")

# ---- DETECT LANGUAGE ----
languages = []
for i, text in enumerate(df[comment_col]):
    lang = detect_language(text)
    languages.append(lang)
    if (i + 1) % 10 == 0:
        print(f"Processed {i + 1}/{len(df)} → {lang}")
    time.sleep(SLEEP_BETWEEN_CALLS)

df["language"] = languages

# ---- SAVE UPDATED FILE ----
path_obj = Path(FILE_PATH)
new_name = path_obj.stem + "____lang.xlsx"
out_path = path_obj.with_name(new_name)
df.to_excel(out_path, index=False)

# ---- PRINT SUMMARY ----
counts = df["language"].value_counts()
print(f"\n📊 Language distribution for RHON dataset:")
print(counts.to_string())
print(f"✅ Saved file with language column → {out_path}")


🔍 Processing RHON dataset: /Users/sushildalavi/Desktop/NLC/WBProj/RHON YT & X for LLM Coding.xlsx
Processed 10/68 → English
Processed 20/68 → English
Processed 30/68 → English
Processed 40/68 → English
Processed 50/68 → English
Processed 60/68 → English

📊 Language distribution for RHON dataset:
language
English    66
Swahili     1
Zulu        1
✅ Saved file with language column → /Users/sushildalavi/Desktop/NLC/WBProj/RHON YT & X for LLM Coding____lang.xlsx
